In [ ]:
import psycopg2
import pandas as pd
import os

dwh = psycopg2.connect( #Подключаемся к БД
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="Kmechte1!"  
)

cursor_dwh = dwh.cursor() #Создаю курсор
today = '01032021' #Дата для сменый файлов (Временно)
directory = os.getcwd() 

In [ ]:
############################################################
#СОЗДАЮ ВРЕМННЫЕ ТАБЛИЦЫ ИЗ DATALAKE 
############################################################

def create_tmp_terminals():
    try:
        cursor_dwh.execute('''
        CREATE TABLE IF NOT EXISTS temp."TERMINALS"(
            terminal_id varchar(5) PRIMARY KEY,
            terminal_type varchar(5),
            terminal_city varchar(20),
            terminal_address varchar(100),
            load_dt date DEFAULT CURRENT_DATE
        )
        ''')
        dwh.commit()
    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()

def create_tmp_transactions():
    try:
        cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS temp."TRANSACTIONS"(
        trans_id varchar(32),
        trans_date date,
        card_num varchar(32),
        oper_type varchar(32),
        amt decimal(10,2),
        oper_result varchar(32),
        terminal varchar(32),
        load_dt date DEFAULT CURRENT_DATE
               )
                ''')
        dwh.commit()
    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()


def create_tmp_pbl():
    try:
        cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS temp."PBL"(
        passport_num varchar(32) PRIMARY KEY,
        entry_dt date,
        load_dt date DEFAULT CURRENT_DATE
               )
                ''')
        dwh.commit()
    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()

def show_table(SchemaName, TableName):
    try:
        cursor_dwh.execute(f'SELECT * FROM {SchemaName}."{TableName}"')

        # 🔹 Заголовки
        col_names = [desc[0] for desc in cursor_dwh.description]
        print(cursor_dwh.description)
        print(" | ".join(col_names))
        print("-" * 50)

        # 🔹 Данные
        rows = cursor_dwh.fetchall()
        if not rows:
            print("(таблица пустая)")
        else:
            for row in rows:
                print(row)

    except Exception as e:
        print("SELECT error:", e)
        dwh.rollback()

create_tmp_transactions()
create_tmp_pbl()
create_tmp_terminals()

############################################################
#ИЗВЛЕКАЕМ ДАННЫЕ ИЗ DATALAKE ВО ВРЕМЕННЫЕ ТАБЛИЦЫ 
############################################################

def file2tmp_pbl():
    try:
        df = pd.read_excel(f'{directory}/passport_blacklist_{today}.xlsx', sheet_name='blacklist')
        for idx, row in df.iterrows():  # перебираем строки
            entry_dt = row['date']   # столбец с датой в Excel
            passport_num = row['passport']  # столбец с паспортом
            cursor_dwh.execute('''
                INSERT INTO temp."PBL"(entry_dt, passport_num)
                VALUES (%s, %s)
            ''', (entry_dt, passport_num))
            dwh.commit()  # фиксируем изменения
    
    except Exception as e:
        print("SELECT error:", e)
        dwh.rollback()

def file2tmp_term():
    try:
        df = pd.read_excel(f'{directory}/terminals_{today}.xlsx', sheet_name='terminals')
        for idx, row in df.iterrows():  # перебираем строки
            terminal_id = row['terminal_id']   
            terminal_type = row['terminal_type'] 			
            terminal_city = row['terminal_city']
            terminal_address = row['terminal_address']
            cursor_dwh.execute('''
                INSERT INTO temp."TERMINALS"(terminal_id, terminal_type, terminal_city, terminal_address)
                VALUES (%s, %s, %s, %s)
            ''', (terminal_id, terminal_type, terminal_city, terminal_address))
            dwh.commit()  # фиксируем изменения

    except Exception as e:
        print("SELECT error:", e)
        dwh.rollback()   

def file2tmp_trans():
    try:
        df = pd.read_csv(f'{directory}/transactions_{today}.txt', sep=';')
        for idx, row in df.iterrows():  # перебираем строки
            trans_id = str(row['transaction_id']) 
            trans_date = row['transaction_date']	
            amt = float(row['amount'].replace(',', '.'))
            card_num = str(row['card_num'])
            oper_type = str(row['oper_type'])
            oper_result = str(row['oper_result'])
            terminal = str(row['terminal'])
            cursor_dwh.execute('''
                INSERT INTO temp."TRANSACTIONS"(trans_id, trans_date, amt, card_num, oper_type, oper_result, terminal)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
            ''', (trans_id, trans_date, amt, card_num, oper_type, oper_result, terminal))
            dwh.commit()  # фиксируем изменения

    except Exception as e:
        print("SELECT error:", e) 

file2tmp_pbl()
file2tmp_term()
file2tmp_trans()

#Теперь данные в схеме временной 

In [309]:
############################################################
#Создаем STG Tables
############################################################
#BLACK_LIST
###########################

def create_pbl_new():
    try:
        cursor_dwh.execute('''
        CREATE TABLE IF NOT EXISTS stg."PBL_new"(
            passport_num varchar(32) PRIMARY KEY,
            entry_dt date
                )
                    ''')
        dwh.commit()

    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()    

def create_pbl_del():
    try:
        cursor_dwh.execute('''
        CREATE TABLE IF NOT EXISTS stg."PBL_del"(
            passport_num varchar(32) PRIMARY KEY,
            entry_dt date
                )
                    ''')
        dwh.commit()

    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()   

def create_pbl_chng():
    try:
        cursor_dwh.execute('''
        CREATE TABLE IF NOT EXISTS stg."PBL_chng"(
            passport_num varchar(32) PRIMARY KEY,
            entry_dt date
                )
                    ''')
        dwh.commit()

    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()   

############################################################
#Создаем FACT Tables
############################################################
#BLACK_LIST
###########################

def create_pbl_fct():
    try:
        cursor_dwh.execute('''
        CREATE TABLE IF NOT EXISTS fct."PBL_t"(
            passport_num varchar(32) PRIMARY KEY,
            entry_dt date
                )
                    ''')
        dwh.commit()

    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()    


############################################################
#Пополнение таблиц STG
############################################################

def tmp2new_pbl():
    try:
        cursor_dwh.execute('''
        INSERT INTO stg."PBL_new"(passport_num, entry_dt)
        SELECT tmp.passport_num, tmp.entry_dt  
        FROM temp."PBL" tmp
        LEFT JOIN fct."PBL_t" fct ON tmp.passport_num=fct.passport_num
        WHERE fct.passport_num IS NULL
            AND tmp.load_dt <> %s
                    ''', (today,))
        dwh.commit()

    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()

def tmp2dlt_pbl():
    try:
        cursor_dwh.execute('''
        INSERT INTO stg."PBL_new"(passport_num, entry_dt)
        SELECT tmp.passport_num, tmp.entry_dt  
        FROM temp."PBL" tmp
        LEFT JOIN fct."PBL_t" fct ON tmp.passport_num=fct.passport_num
        WHERE fct.passport_num IS NULL
            AND tmp.load_dt <> %s
                    ''', (today,))
        dwh.commit()
    except Exception as e:
        print("CREATE TABLE error:", e)
        dwh.rollback()
    

In [ ]:
#Execution tab
#create_pbl_new()
#create_pbl_del()
#create_pbl_chng()
#create_pbl_fct()

In [ ]:
tmp2new_pbl()
tmp2dlt_pbl()

CREATE TABLE error: date/time field value out of range: "01032021"
LINE 7:             AND tmp.load_dt <> '01032021'
                                       ^
HINT:  Perhaps you need a different "DateStyle" setting.

CREATE TABLE error: date/time field value out of range: "01032021"
LINE 7:             AND tmp.load_dt <> '01032021'
                                       ^
HINT:  Perhaps you need a different "DateStyle" setting.



In [ ]:


def create_terminals_new():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TERMINALS_new"(
        terminal_id varchar(5) PRIMARY KEY,
        terminal_type varchar(5),
        terminal_city varchar(20),
        terminal_address varchar(100),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date
               )
                ''')
    
def create_terminals_del():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TERMINALS_del"(
        terminal_id varchar(5) PRIMARY KEY,
        terminal_type varchar(5),
        terminal_city varchar(20),
        terminal_address varchar(100),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date
               )
                ''')
            
def create_terminals_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TERMINALS_del"(
        terminal_id varchar(5) PRIMARY KEY,
        terminal_type varchar(5),
        terminal_city varchar(20),
        terminal_address varchar(100),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date
               )
                ''')

def create_clients_new():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CLIENTS_new"(
        client_id varchar(32) PRIMARY KEY,
        last_name varchar(32),
        first_name varchar(32),
        patrinymic varchar(32),
        date_of_birth date,
        paport_num varchar(15),
        passpot_valid_to date,
        phone varchar(15),
        effective_from date,
        effective_to date,
        deleted_flg binary
               )
                ''')

def create_clients_del():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CLIENTS_del"(
        client_id varchar(32) PRIMARY KEY,
        last_name varchar(32),
        first_name varchar(32),
        patrinymic varchar(32),
        date_of_birth date,
        paport_num varchar(15),
        passpot_valid_to date,
        phone varchar(15),
        effective_from date,
        effective_to date,
        deleted_flg binary
               )
                ''')

def create_clients_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CLIENTS_chng"(
        client_id varchar(32) PRIMARY KEY,
        last_name varchar(32),
        first_name varchar(32),
        patrinymic varchar(32),
        date_of_birth date,
        paport_num varchar(15),
        passpot_valid_to date,
        phone varchar(15),
        effective_from date,
        effective_to date,
        deleted_flg binary
               )
                ''')

def create_accounts_new():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "ACCAUNTS_new"(
        accaunt_num varchar(32) PRIMARY KEY,
        valid_to date,
        client varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (client) REFERENCES CLIENTS(client_id)
               )
                ''')

def create_accounts_del():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "ACCAUNTS_del"(
        accaunt_num varchar(32) PRIMARY KEY,
        valid_to date,
        client varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (client) REFERENCES CLIENTS(client_id)
               )
                ''')
        
def create_accounts_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "ACCAUNTS_chng"(
        accaunt_num varchar(32) PRIMARY KEY,
        valid_to date,
        client varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (client) REFERENCES CLIENTS(client_id)
               )
                ''')

def create_cards_new():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CARDS_new"(
        card_num varchar(32) PRIMARY KEY,
        accaunt_num varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (accaunt_num) REFERENCES ACCAUNTS(accaunt_num)
               )
                ''')

def create_cards_del():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CARDS_del"(
        card_num varchar(32) PRIMARY KEY,
        accaunt_num varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (accaunt_num) REFERENCES ACCAUNTS(accaunt_num)
               )
                ''')

def create_cards_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CARDS_chng"(
        card_num varchar(32) PRIMARY KEY,
        accaunt_num varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (accaunt_num) REFERENCES ACCAUNTS(accaunt_num)
               )
                ''')

def create_transactions_new():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TRANSACTIONS_new"(
        trans_id varchar(32),
        trans_date date,
        card_num varchar(32),
        order_type varchar(32),
        amt decimal(10,2),
        oper_result varchar(32),
        terminal varchar(32),
        effective_from date,
        effective_to date,
        deleted_flg binary,
        FOREIGN KEY (card_num) REFERENCES CARDS(card_num),
        FOREIGN KEY (terminal) REFERENCES ACCAUNTS(terminal_id)
               )
                ''')
        
def create_transactions_del():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TRANSACTIONS_del"(
        trans_id varchar(32),
        trans_date date,
        card_num varchar(32),
        order_type varchar(32),
        amt decimal(10,2),
        oper_result varchar(32),
        terminal varchar(32),
        effective_from date,
        effective_to date,
        deleted_flg binary,
        FOREIGN KEY (card_num) REFERENCES CARDS(card_num),
        FOREIGN KEY (terminal) REFERENCES ACCAUNTS(terminal_id)
               )
                ''')

def create_transactions_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TRANSACTIONS_chng"(
        trans_id varchar(32),
        trans_date date,
        card_num varchar(32),
        order_type varchar(32),
        amt decimal(10,2),
        oper_result varchar(32),
        terminal varchar(32),
        effective_from date,
        effective_to date,
        deleted_flg binary,
        FOREIGN KEY (card_num) REFERENCES CARDS(card_num),
        FOREIGN KEY (terminal) REFERENCES ACCAUNTS(terminal_id)
               )
                ''')

############################################################
#Пополнение таблиц STG
############################################################

def tmp2new_pbl():
    cursor_dwh.execute('''
    INSERT INTO "PBL_new"(passport_num, entry_dt)
    SELECT tmp.passport, date  
    FROM PBL_temp tmp
    LEFT JOIN "PBL_fct" fct
        ON tmp.passport=fct.passport_num
    WHERE fct.passport_num IS NULL
                ''')

def tmp2dlt_pbl():
    cursor_dwh.execute('''
    INSERT INTO "PBL_new"(passport_num, entry_dt)
    SELECT tmp.passport, date  
    FROM PBL_temp tmp
    LEFT JOIN "PBL_fct" fct
        ON tmp.passport=fct.passport_num
    WHERE tmp.passport_num IS NULL
                ''')

def tmp2new_cards():
    cursor_dwh.execute('''
    INSERT INTO "CARDS_new"(card_num, entry_dt)
    SELECT tmp.passport, date  
    FROM PBL_temp tmp
    LEFT JOIN "PBL_fct" fct
        ON tmp.passport=fct.passport_num
    WHERE fct.passport_num IS NULL
                ''')

def tmp2dlt_card():
    cursor_dwh.execute('''
    INSERT INTO "PBL_new"(passport_num, accaunt_num, create_dt, )
    SELECT tmp.passport, date  
    FROM PBL_temp tmp
    LEFT JOIN "PBL_fct" fct
        ON tmp.passport=fct.passport_num
    WHERE tmp.passport_num IS NULL
                ''')

def create_cards_chng():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CARDS_chng"(
        card_num varchar(32) PRIMARY KEY,
        accaunt_num varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (accaunt_num) REFERENCES ACCAUNTS(accaunt_num)
               )
                ''')




In [ ]:
############################################################
#Создаем STAGING Tables
############################################################
def create_table_terminals():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS staging."TERMINALS_stg"(
        terminal_id varchar(5) PRIMARY KEY,
        terminal_type varchar(5),
        terminal_city varchar(20),
        terminal_address varchar(100),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date
               )
                ''')

def create_table_clients():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CLIENTS_stg"(
        client_id varchar(32) PRIMARY KEY,
        last_name varchar(32),
        first_name varchar(32),
        patrinymic varchar(32),
        date_of_birth date,
        paport_num varchar(15),
        passpot_valid_to date,
        phone varchar(15),
        effective_from date,
        effective_to date,
        deleted_flg binary
               )
                ''')

def create_table_accounts():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "ACCAUNTS_stg"(
        accaunt_num varchar(32) PRIMARY KEY,
        valid_to date,
        client varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (client) REFERENCES CLIENTS(client_id)
               )
                ''')

def create_table_cards():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "CARDS_stg"(
        card_num varchar(32) PRIMARY KEY,
        accaunt_num varchar(32),
        create_dt date DEFAULT (strftime('%d%m%Y', 'now')),
        update_dt date,
        FOREIGN KEY (accaunt_num) REFERENCES ACCAUNTS(accaunt_num)
               )
                ''')

def create_table_transactions():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "TRANSACTIONS_stg"(
        trans_id varchar(32),
        trans_date date,
        card_num varchar(32),
        order_type varchar(32),
        amt decimal(10,2),
        oper_result varchar(32),
        terminal varchar(32),
        effective_from date,
        effective_to date,
        deleted_flg binary,
        FOREIGN KEY (card_num) REFERENCES CARDS(card_num),
        FOREIGN KEY (terminal) REFERENCES ACCAUNTS(terminal_id)
               )
                ''')

def create_table_pbl():
    cursor_dwh.execute('''
    CREATE TABLE IF NOT EXISTS "PBL_stg"(
        passport_num varchar(32) PRIMARY KEY,
        entry_dt date
               )
                ''')

In [ ]:
transaction_id;transaction_date;amount;card_num;oper_type;oper_result;terminal
print(df)
#for row_index, row_data in df.iterrows():
 #   print("Index:", row_index)
 #   print("Data:", row_data)

       transaction_id     transaction_date   amount             card_num  \
0         43845789347  2021-03-01 00:00:01  1046,40  4513 5880 2369 1799   
1         43845789803  2021-03-01 00:00:05  6254,20  4422 8510 8242 3474   
2         43845790032  2021-03-01 00:00:12  1413,90  4600 5574 2101 5919   
3         43845790080  2021-03-01 00:00:14  1200,00  2343 9229 2085 3223   
4         43845790198  2021-03-01 00:00:22  7000,00  5757 4476 6224 7806   
...               ...                  ...      ...                  ...   
15645     43853690837  2021-03-01 23:59:23  8500,00  4714 5486 8211 6044   
15646     43853691408  2021-03-01 23:59:31  2614,90  2297 5825 3755 3296   
15647     43853692395  2021-03-01 23:59:39  5513,90  4709 4592 6306 2366   
15648     43853692879  2021-03-01 23:59:49  4608,00  5901 3598 1337 3802   
15649     43853693830  2021-03-01 23:59:56  6500,00  4243 6488 4324 6675   

      oper_type oper_result terminal  
0       PAYMENT     SUCCESS    P5456  
1       P

In [ ]:


#show_table("temp","PBL")




file2tmp_pbl()

#file2sql('excel','terminals','terminals','TERMINALS_temp')
#file2sql('excel','passport_blacklist','blacklist','PBL_temp')
#file2sql('csv','transactions','transactions','TRANSACTIONS_temp')


In [ ]:
############################################################
#FROM CSV 2 SQL
############################################################

def file2sql(FileType, FilePrefix,Sheet_name, TableName2Add):
    if FileType == 'csv':
        df = pd.read_csv(f'{directory}/{FilePrefix}_{today}.txt', sep=';')
    elif FileType == 'excel':
        df = pd.read_excel(f'{directory}/{FilePrefix}_{today}.xlsx', sheet_name=Sheet_name)
    else:
        print("I dont know this FileType")
    df.to_sql(TableName2Add, con=dwh, if_exists='replace', index = False)

def show_table(TableName):
    cursor_dwh.execute(f'SELECT * FROM "{TableName}"')
    for row in cursor_dwh.fetchall():
        print(row)

file2sql('excel','terminals','terminals','TERMINALS_temp')
file2sql('excel','passport_blacklist','blacklist','PBL_temp')
file2sql('csv','transactions','transactions','TRANSACTIONS_temp')


/var/folders/b8/6sfw7zx15ys4hs59t026r4200000gn/T/ipykernel_25183/186669436.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df.to_sql(TableName2Add, con=dwh, if_exists='replace', index = False)


DatabaseError: Execution failed on sql '
        SELECT
            name
        FROM
            sqlite_master
        WHERE
            type IN ('table', 'view')
            AND name=?;
        ': syntax error at or near ";"
LINE 8:             AND name=?;
                              ^


In [96]:
show_table('PBL_temp')

('2021-03-01 00:00:00', '9933 106914')
('2021-03-01 00:00:00', '6915 535193')
('2021-03-01 00:00:00', '5385 691850')
('2021-03-01 00:00:00', '8683 912237')
('2021-03-01 00:00:00', '8340 525086')
('2021-03-01 00:00:00', '3110 700486')
('2021-03-01 00:00:00', '3384 214650')


In [110]:
create_pbl_new()
create_table_pbl()
tmp2new_pbl()



IntegrityError: UNIQUE constraint failed: PBL_new.passport_num

In [112]:
show_table('PBL_new')

('9933 106914', '2021-03-01 00:00:00')
('6915 535193', '2021-03-01 00:00:00')
('5385 691850', '2021-03-01 00:00:00')
('8683 912237', '2021-03-01 00:00:00')
('8340 525086', '2021-03-01 00:00:00')
('3110 700486', '2021-03-01 00:00:00')
('3384 214650', '2021-03-01 00:00:00')


source --> temptable --> new_data --> hist_data

In [ ]:





############################################################
#SELECT
############################################################



def show_lake_Term():
    cursor_lake.execute('SELECT * FROM "TERMINALS"')
    for row in cursor_lake.fetchall():
        print(row) 

def show_stage_BL():
    cursor_stage.execute('SELECT * FROM "passport blacklist"')
    for row in cursor_stage.fetchall():
        print(row)  

def show_stage_Term():
    cursor_stage.execute('SELECT * FROM "TERMINALS"')
    for row in cursor_stage.fetchall():
        print(row) 







In [ ]:
excel2sql()

In [46]:
directory = os.getcwd()

def excel2sql(FilePrefix,Sheet_name, TableName2Add):
    print(f'{directory}/{FilePrefix}.xlsx, sheet_name={Sheet_name}')

excel2sql('terminals', 'terminals', 'fff')

/Users/maksim/Documents/Coding/mssannikov.github.io/Data_Engenireeng/Increment_download/terminals.xlsx, sheet_name=terminals


In [ ]:
############################################################
#Поиск данных
############################################################



'''create_table_terminals
create_table_clients()
create_table_accounts()
create_table_cards()
create_table_transactions()
create_table_passport_blk'''


terminal_data = pd.read_excel(f'{directory}/terminals_{today}.xlsx', sheet_name='terminals')
bl_data = pd.read_excel(f'{directory}/passport_blacklist_{today}.xlsx', sheet_name='blacklist')
trnsactions_data= pd.read_csv(f'{directory}/transactions_{today}.txt', sep=';')

print(terminal_data)

# Specify the directory path




    terminal_id terminal_type    terminal_city  \
0         A1096           ATM         Кемерово   
1         A1099           ATM           Баймак   
2         A1203           ATM      Стерлитамак   
3         A1553           ATM          Воронеж   
4         A1641           ATM           Москва   
..          ...           ...              ...   
145       P9862           POS      Новокузнецк   
146       P9889           POS           Усмань   
147       P9974           POS            Томск   
148       P9977           POS  Нижний Новгород   
149       P9978           POS         Знаменск   

                                 terminal_address  
0    г. Кемерово, 1-й Электрозаводский пер., д. 3  
1            г. Баймак, Б. Лёвшинский пер., д. 37  
2      г. Стерлитамак, Электрозаводская ул., д. 3  
3              г. Воронеж, пр. Энтузиастов, д. 44  
4          г. Москва, 1-й Южнопортовый пр., д. 26  
..                                            ...  
145     г. Новокузнецк, Б. Лёвшинск

In [29]:
############################################################
#Вставка данных
############################################################

for dt in range(len(bl)):
    # Преобразуем дату в строку формата YYYY-MM-DD
    date_str = pd.to_datetime(bl['date'][dt]).strftime('%Y-%m-%d')
    passport = str(bl['passport'][dt])
    
    cursor.execute('''
        INSERT INTO "PASSPORT BLACKLIST" (passport_num, entry_dt)
        VALUES (?, ?)
    ''', (passport, date_str))

for dt in range(len(ter)):  
    cursor.execute('''
        INSERT INTO "TERMINALS" (terminal_id, terminal_type, terminal_city, terminal_address)
        VALUES (?, ?, ?, ?)
    ''', (ter['terminal_id'][dt], ter['terminal_type'][dt], ter['terminal_city'][dt], ter['terminal_address'][dt]))



IntegrityError: UNIQUE constraint failed: PASSPORT BLACKLIST.passport_num

In [ ]:
############################################################
#Показать данные в таблице
############################################################

showTerm()
#print(type(ter['terminal_id'][1]))
     # ter['terminal_type'], ter['terminal_city'], ter['terminal_address'])

# List all files and directories
#contents = os.listdir(directory)
#print(contents)

## 

#
# Сохраняем изменения и закрываем соединение
#conn.commit()#

   
    
#showUsers()


def proc_lake2stage_BL():
    cursor_lake.execute('''
DELIMITER $$

CREATE PROCEDURE IF NOT EXISTS proc_lake2stage_BL()
BEGIN
    INSERT INTO create_stage_table_passport_blk.target_table (id, value, created_at)
    SELECT 
        s.id,
        s.value,
        s.created_at
    FROM db_source.source_table AS s
    RIGHT JOIN db_target.target_table AS t
        ON s.id = t.id
    WHERE t.id IS NULL;   -- значит, записи нет в целевой таблице
END $$

DELIMITER ;
''')

('A1096', 'ATM', 'Кемерово', 'г. Кемерово, 1-й Электрозаводский пер., д. 3', 9122025, None)
('A1099', 'ATM', 'Баймак', 'г. Баймак, Б. Лёвшинский пер., д. 37', 9122025, None)
('A1203', 'ATM', 'Стерлитамак', 'г. Стерлитамак, Электрозаводская ул., д. 3', 9122025, None)
('A1553', 'ATM', 'Воронеж', 'г. Воронеж, пр. Энтузиастов, д. 44', 9122025, None)
('A1641', 'ATM', 'Москва', 'г. Москва, 1-й Южнопортовый пр., д. 26', 9122025, None)
('A1642', 'ATM', 'Новокуйбышевск', 'г. Новокуйбышевск, 1-й Щипковский пер., д. 19', 9122025, None)
('A1882', 'ATM', 'Иркутск', 'г. Иркутск, Юровская ул., д. 3', 9122025, None)
('A2293', 'ATM', 'Краснодар', 'г. Краснодар, Южный пр., д. 20', 9122025, None)
('A2674', 'ATM', 'Истра', 'г. Истра, ул. Юности, д. 8', 9122025, None)
('A2758', 'ATM', 'Жигулёвск', 'г. Жигулёвск, М. Юшуньская ул., д. 25', 9122025, None)
('A2822', 'ATM', 'Томск', 'г. Томск, Юрловский пр., д. 22', 9122025, None)
('A3174', 'ATM', 'Москва', 'г. Москва, 1-й Щемиловский пер., д. 8', 9122025, None